In [1]:
!pip install -q sentence-transformers transformers accelerate torch
!pip install -q supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import json
import os
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

In [4]:
knowledge_path = "/content/drive/MyDrive/Hackathon AI /knowledge_base/knowledge_base.json"

with open(knowledge_path, "r", encoding="utf-8") as f:
    documents = json.load(f)

print("Jumlah Dokumen:", len(documents))

Jumlah Dokumen: 22760


load embedding model

In [5]:
model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    trust_remote_code=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

In [6]:
import torch

print("GPU tersedia :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("Model Device :", model.device)

GPU tersedia : True
GPU : Tesla T4
Model Device : cuda:0


test

In [7]:
sample = documents[0]["content"]

embedding = model.encode(sample)

print("Dimensi Embedding :", len(embedding))

Dimensi Embedding : 1024


generate semua embedding

In [9]:
import json
import torch
from tqdm import tqdm

output_path = "/content/drive/MyDrive/Hackathon AI /knowledge_base/knowledge_base_embedding.json"

torch.cuda.empty_cache()

batch_size = 4

with open(output_path, "w", encoding="utf-8") as f:

    f.write("[\n")

    first = True

    for i in tqdm(range(0, len(documents), batch_size)):

        batch = documents[i:i+batch_size]

        texts = [doc["content"] for doc in batch]

        embeddings = model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        for doc, emb in zip(batch, embeddings):

            item = {
                "content": doc["content"],
                "metadata": doc.get("metadata", {}),
                "embedding": emb.tolist()
            }

            if not first:
                f.write(",\n")

            json.dump(item, f, ensure_ascii=False)

            first = False

        torch.cuda.empty_cache()

    f.write("\n]")

print("Embedding selesai disimpan.")
print(output_path)

100%|██████████| 5690/5690 [22:35<00:00,  4.20it/s]

Embedding selesai disimpan.
/content/drive/MyDrive/Hackathon AI /knowledge_base/knowledge_base_embedding.json


In [10]:
import json

file_path = "/content/drive/MyDrive/Hackathon AI /knowledge_base/knowledge_base_embedding.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Jumlah data :", len(data))
print("Keys :", data[0].keys())

Jumlah data : 22760
Keys : dict_keys(['content', 'metadata', 'embedding'])
